# DapurDana MVP - Predictive Pricing AI

Notebook ini menyiapkan pipeline minimum untuk data harga pangan harian: cleaning, interpolasi time-series, feature engineering volatilitas mingguan, baseline ARIMA/SARIMAX, dan evaluasi RMSE/MAPE. Dataset contoh berada di `../datasets/raw/sample_food_prices.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

DATA_PATH = Path('../datasets/raw/sample_food_prices.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.head()

In [ ]:
def clean_commodity(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values('date').copy()
    group['price_interpolated'] = group['price'].interpolate(method='linear')
    group['price_interpolated'] = group['price_interpolated'].ffill().bfill()
    group['daily_return'] = group['price_interpolated'].pct_change().fillna(0)
    group['weekly_volatility'] = group['daily_return'].rolling(7, min_periods=2).std().fillna(0)
    return group

clean_df = df.groupby(['region', 'commodity'], group_keys=False).apply(clean_commodity)
clean_df.to_csv('../datasets/processed/sample_food_prices_clean.csv', index=False)
clean_df.head()

In [ ]:
def evaluate_forecast(series: pd.Series, test_size: int = 3):
    train = series.iloc[:-test_size]
    test = series.iloc[-test_size:]

    model = SARIMAX(
        train,
        order=(1, 1, 1),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False)
    forecast = fitted.forecast(steps=test_size)

    rmse = mean_squared_error(test, forecast, squared=False)
    mape = mean_absolute_percentage_error(test, forecast) * 100
    return rmse, mape, forecast

results = []
for commodity, group in clean_df.groupby('commodity'):
    series = group.sort_values('date').set_index('date')['price_interpolated']
    rmse, mape, forecast = evaluate_forecast(series)
    results.append({'commodity': commodity, 'rmse': rmse, 'mape': mape, 'next_prediction': forecast.iloc[-1]})

pd.DataFrame(results).sort_values('mape')

## Catatan Pengembangan

- Untuk MVP cepat, SARIMAX/ARIMA dipakai sebagai baseline model yang mudah dievaluasi.
- LSTM dapat ditambahkan setelah dataset historis minimal 1-2 tahun tersedia agar sequence training tidak overfit.
- Output model produksi harus diekspos sebagai endpoint Azure ML dengan bentuk response yang sama seperti `/api/predict` backend Node.js.